# Arbitrary Downscaling of a Spline
We display a panel of four figures.

*   The top figure contains a periodic spline $f_{0}$ at nominal scale, with a specified period $K_{0},$ degree $n_{0},$ and delay $\delta_{0},$ the spline coefficients of $f_{0}$ being realizations of a Cauchy random variable.
*   The upper-middle figure contains $f_{K_{0}\downarrow m},$ which is a version of $f_{0}$ whose features have been downscaled (*i.e.*, minified) by the positive integer factor $m\in{\mathbb{N}}+1.$ Its period is the integer $K=\frac{K_{0}}{\gcd(K_{0},m)},$ which may or may not be smaller than $K_{0}$ because it is the *features* that are minified, not necessarily the period. The least-squares criterion $J=\frac{1}{2}\,\int_{0}^{m\,K}\,\left(f_{K_{0}\downarrow m}(\frac{x}{m})-f_{0}(x)\right)^{2}\,{\mathrm{d}}x$ is minimized.
*   The lower-middle figure contains $\frac{m}{\gcd(K_{0},m)}$ replicates of $f_{0}$ in thick gray, along with $f_{\left(K_{0}\downarrow m\right)\uparrow m}$ in colors. The latter is the exact re-upscaling of $f_{K_{0}\downarrow m}$ and is a function that happens to be again at the nominal scale in terms of features, and that is $\left(\frac{m}{\gcd(K_{0},m)}\,K_{0}\right)$-periodic.
*   The bottom figure takes relevance when $m$ does not divide $K_{0}$ entirely, in which case the pseudo-periods of the minified $f_{K_{0}\downarrow m}$ have the non-integer length $\frac{K_{0}}{m}\not\in{\mathbb{N}}.$ It contains a superposition of the pseudo-periods of $f_{\left(K_{0}\downarrow m\right)\uparrow m}$ that overlay the nominal $f_{0}.$

For $f_{0}$ and $f_{K_{0}\downarrow m},$ the samples at the integers are indicated by circles and stem lines, while the knots are shown as black dots. The boundaries of one period are highlighted in red. Finally, we plot each (possibly noninteger) pseudo-period of length $\frac{K_{0}}{m}$ in a different color.

We print a numeric estimate of the integral (over $\frac{m}{\gcd(K_{0},m)}$ nominal periods $K_{0}$) of the product between $f_{\left(K_{0}\downarrow m\right)\uparrow m}$ and the residue $\left(f_{\left(K_{0}\downarrow m\right)\uparrow m}-f_{0}\right).$ This scalar product is expected to vanish, and this is precisely what happens, up to numerical accuracy.

Finally, we print the same quantity but we avoid numeric estimates, relying instead on direct computations that are based on convolutions.

In [ ]:
# Load the required libraries
from IPython.display import display
from IPython.display import Math
import ipywidgets as widgets
import math
import matplotlib.pyplot as plt
import numpy as np
import scipy
import warnings

import splinekit as sk # This library

# Setup
max_period = 24 # maximal period
max_degree = 9 # Maximal spline degree
max_delay = 3.0 # Maximal absolute delay
max_minif = 5 # Maximal minification factor

# Initialize the generator of random numbers
rng = np.random.default_rng()

# Random periodic cubic spline
f0 = sk.PeriodicSpline1D.from_spline_coeff(rng.standard_cauchy(12), degree = 3)

# Plot
def update_plot (
    period0 = 10,
    minif = 3,
    degree0 = 3,
    delay0 = 0.0,
    degree = 1,
    delay = 0.0
):
    global f0

    # Update of the spline
    if f0.period != period0:
        f0 = sk.PeriodicSpline1D.from_spline_coeff(
            rng.standard_cauchy(period0),
            degree = f0.degree
        )
    f0.degree = degree0
    f0.delay = delay0

    # Downscaling
    fm = f0.downscaled_projected(minification = minif, degree = degree, delay = delay)

    # Plots
    (fig, (ax1, ax2, ax3, ax4)) = plt.subplots(nrows = 4)

    # First panel, spline at the nominal scale
    f0.plot((fig, ax1), plotpoints = 200 + 1)

    # Second panel, minified spline
    image = fm.image()
    if isinstance(image, sk.interval.Singleton):
        plotrange = sk.interval.Closed((image.midpoint - 0.5, image.midpoint + 0.5))
    else:
        plotrange = sk.interval.Closed((
            image.midpoint - 0.55 * image.diameter,
            image.midpoint + 0.55 * image.diameter
        ))
    p = minif // math.gcd(period0, minif) # Number of pseudo periods
    c = p - 1 # Rightmost color
    fm.plot(
        (fig, ax2),
        plotpoints = 20 + 1,
        plotdomain = sk.interval.OpenClosed((-1.0, 0.0)),
        plotrange = plotrange,
        curve_fmt = "-C" + str((c % p) % 10)
    ) # Left margin
    c = 0 # Color
    for k in range(p): # Plot each pseudo-period with a different color
        fm.plot(
            (fig, ax2),
            plotpoints = 200 // minif + 1,
            plotdomain = sk.interval.Closed(
                (k * period0 / minif, (k + 1) * period0 / minif)
            ),
            plotrange = plotrange,
            curve_fmt = "-C" + str((c % p) % 10)
        ) # Pseudo-period
        c += 1
    c = 0 # Leftmost color
    fm.plot(
        (fig, ax2),
        plotpoints = 20 + 1,
        plotdomain = sk.interval.ClosedOpen((fm.period, fm.period + 1.0)),
        plotrange = plotrange,
        curve_fmt = "-C" + str((c % p) % 10)
    ) # Right margin

    # Third panel, Replicates of the nominal spline and magnification of the minified spline
    f00 = sk.PeriodicSpline1D.from_spline_coeff(
        np.tile(f0.spline_coeff, minif // math.gcd(period0, minif)),
        degree = degree0,
        delay = delay0
    ) # Repeated periods of the nominal spline
    fmu = fm.upscaled(magnification = minif) # Magnification of the minified spline
    # Dynamic range
    image = {f0.image(), fmu.image()}
    plotrange = sk.interval.Interval.enclosure(image)
    if isinstance(plotrange, sk.interval.Singleton):
        plotrange = sk.interval.Closed((
            plotrange.midpoint - 0.5,
            plotrange.midpoint + 0.5
        ))
    else:
        plotrange = sk.interval.Closed((
            plotrange.midpoint - 0.55 * plotrange.diameter,
            plotrange.midpoint + 0.55 * plotrange.diameter
        ))
    f00.plot(
        (fig, ax3),
        plotpoints = 200 + 1,
        plotdomain = sk.interval.ClosedOpen((-minif, f00.period + minif)),
        plotrange = plotrange,
        curve_fmt = "#e0e0e0",
        curve_lw = 7.0,
        curve_markerfmt = " ",
        curvestem_linefmt = "None",
        knot_marker = " ",
        periodbound_markerfmt = " ",
        periodboundstem_linefmt = "None"
    )
    c = 0 # Color
    for k in range(p): # Plot each pseudo-period with a different color
        fmu.plot(
            (fig, ax3),
            plotpoints = 200,
            plotdomain = sk.interval.Closed((k * period0, (k + 1) * period0)),
            plotrange = plotrange,
            curve_fmt = "-C" + str((c % p) % 10),
            curve_markerfmt = " ",
            curvestem_linefmt = "None",
            knot_marker = " ",
            periodbound_markerfmt = " ",
            periodboundstem_linefmt = "None"
        ) # Translated pseudo-period
        c += 1

    # Fourth panel, Superposition of pseudo-periods
    f0.plot(
        (fig, ax4),
        plotpoints = 200 + 1,
        plotrange = plotrange,
        curve_fmt = "#e0e0e0",
        curve_lw = 7.0,
        curve_markerfmt = " ",
        curvestem_linefmt = "None",
        knot_marker = " ",
        periodbound_markerfmt = " ",
        periodboundstem_linefmt = "None"
    )
    c = p - 1 # Color
    for k in range(p): # Plot each pseudo-period with a different color
        fm.delayed_by(-(p - 1 - k) * period0 / minif).upscaled(magnification = minif).plot(
            (fig, ax4),
            plotpoints = 200,
            plotdomain = sk.interval.Closed((0, period0)),
            plotrange = plotrange,
            curve_fmt = "-C" + str((c % p) % 10),
            curve_markerfmt = " ",
            curvestem_linefmt = "None",
            knot_marker = " ",
            periodbound_markerfmt = " ",
            periodboundstem_linefmt = "None"
        ) # Translated pseudo-period
        c -= 1

    # Final display
    plt.show()

    # Numeric estimate of the scalar product
    def integrand (
        x
    ):
        return(fmu.at(x) * (fmu.at(x) - f00.at(x)))
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        djn = scipy.integrate.quad(
            integrand,
            0,
            fmu.period,
            points = np.concatenate((fmu.get_knots(), f00.get_knots())),
            limit = fmu.period + f00.period + 1
        )
    display(Math(
        r"""
        \int_{{0}}^{{{0:}}}\,
        f_{{\left({1:}\downarrow{2:}\right)\uparrow{2:}}}(x)\,
        \left(f_{{\left({1:}\downarrow{2:}\right)\uparrow{2:}}}(x)-f_{{0}}(x))\right)\,
        {{\mathrm{{d}}}}x={3:.2E}
        """.format(fmu.period, period0, minif, djn[0])
    ))

    # Convolution-based scalar product
    fmuv = fmu.mirrored()
    djc = (sk.PeriodicSpline1D.convolve(fmuv, fmu).at(0) -
        sk.PeriodicSpline1D.convolve(fmuv, f00).at(0))
    display(Math(
        r"""
        \left(f_{{\left({0:}\downarrow{1:}\right)\uparrow{1:}}}^{{\vee}}*
        f_{{\left({0:}\downarrow{1:}\right)\uparrow{1:}}}\right)(0)-
        \left(f_{{\left({0:}\downarrow{1:}\right)\uparrow{1:}}}^{{\vee}}*
        f_{{0}}\right)(0)={2:.2E}
        """.format(period0, minif, djc)
    ))

# Interaction
widgets.interactive(
    update_plot,
    period0 = (1, max_period),
    minif = (1, max_minif),
    degree0 = (0, max_degree),
    delay0 = (-max_delay, max_delay, 0.05),
    degree = (0, max_degree),
    delay = (-max_delay, max_delay, 0.05)
)
